# Portable input staging
Run once from the session2 project directory. Public downloads require internet; on an offline Hub stage the exact files first. This helper was prepared after the measured experiment and is not its executed inference source.

In [ ]:
from pathlib import Path
import os, json, shutil, hashlib, sys, importlib.metadata, urllib.request
from huggingface_hub import snapshot_download
ROOT=Path.cwd()
OWN=ROOT/'llm4rec_s2_rq2'
assert (OWN/'inputs/starter.py').is_file(), 'Run from Rustam_Shangareev/session2'
archive=Path(os.environ['FAR_TRANS_ARCHIVE']).expanduser().resolve()
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='d0d6cfe027732bab45fdcf9af80eff4fefddc789aedae9bdce0d3d6cfd05cfb6'
dest=OWN/'inputs/FAR-Trans.zip'
if not dest.exists():shutil.copyfile(archive,dest)
else:assert dest.read_bytes()==archive.read_bytes()
shared=OWN/'shared_contexts';shared.mkdir(exist_ok=True)
base='https://raw.githubusercontent.com/Madlyn-Manneh/llm4recsys-final-project/31e7a2545f8f156f630d193ae9583828bbbdd7e5/Madlyn_Manneh/session2/outputs/baseline/'
expected={'contexts.jsonl':'a505745968316fb96f03a453726878d84501965a12482e81b54b8bf468a1093e','protocol.json':'df787347c013740822aeaa2ea17688e975fe46f608f14aea53235ac76fe58f99'}
for name,digest in expected.items():
    p=shared/name
    if not p.exists():
        with urllib.request.urlopen(base+name,timeout=60) as response:data=response.read()
        assert hashlib.sha256(data).hexdigest()==digest,name
        p.write_bytes(data)
    assert hashlib.sha256(p.read_bytes()).hexdigest()==digest,name
shutil.copyfile(OWN/'shared-context-provenance.json',shared/'provenance.json')
# The public snapshot is staged locally; the unchanged inference notebook verifies every file canonically.
model=OWN/'model_snapshot_70d_v1'
if not (model/'model-00002-of-00002.safetensors').is_file():
    snapshot_download(repo_id='Qwen/Qwen3-1.7B',revision='70d244cc86ccca08cf5af4e1e306ecf908b1ad5e',local_dir=str(model),token=False,allow_patterns=['*.json','*.safetensors','*.txt','.gitattributes','README.md'],etag_timeout=60)
# This describes this reproduction environment. It is not a claim to repeat the original transfer failure.
versions={n:importlib.metadata.version(n) for n in ['torch','transformers','huggingface-hub']}
(OWN/'offline_environment.json').write_text(json.dumps({'python':sys.version,**versions,'installation':'current Python environment; offline during inference','purpose':'portable reproduction of the archived experiment'},indent=2)+'\n')
print('Pinned local data, model and shared contexts are ready.')
